In [27]:
# 3. Current User Query: Woh specific sawal jis ka model ko abhi jawab dena hai.
user_query = "How much oil demand could electric vehicles displace by 2030 and over the 2030-2050 period?"

## RETRIEVING AS PER USER CURRENT QUERY.

In [28]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
import ollama
import uuid

load_dotenv()  # Load environment variables from .env file

#LOADING DOCUMENT 
pdf_path = r"C:\Users\abubakar\Desktop\agentic_notebook\00_data\parsing_data\evs_oil_price_shock.pdf" 
print(f"Loading document from {pdf_path}...")
loader = PyPDFLoader(pdf_path)
docs = loader.load()


# 2. Chunking with LangChain
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)

print(f"Successfully split the PDF into {len(splits)} chunks!")

# CREATING EMBEDDING AND VECTOR STORE


# 3. Embedding and Storing in Chroma Vector Store via Ollama
print("Embedding chunks into Chroma Vector Store...")
embeddings = OllamaEmbeddings(model="all-minilm:latest")

# Generate a unique ID for every single chunk
unique_ids = [str(uuid.uuid4()) for _ in range(len(splits))]

# Pass the unique IDs into the database so it can't overwrite them
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    ids=unique_ids 
)


stored_data = vectorstore.get()
print(f"Total chunks successfully stored: {len(stored_data['ids'])}")

found_chunks = vectorstore.similarity_search(user_query, k=3)


Loading document from C:\Users\abubakar\Desktop\agentic_notebook\00_data\parsing_data\evs_oil_price_shock.pdf...


incorrect startxref pointer(1)
parsing for Object Streams


Successfully split the PDF into 98 chunks!
Embedding chunks into Chroma Vector Store...
Total chunks successfully stored: 294


In [29]:
found_chunks

[Document(metadata={'page': 7, 'trapped': '/False', 'page_label': '8', 'creator': '(unspecified)', 'total_pages': 15, 'producer': 'ReportLab PDF Library - (opensource)', 'subject': '(unspecified)', 'creationdate': '2026-04-18T21:49:03+05:00', 'author': '(anonymous)', 'moddate': '2026-04-18T21:49:03+05:00', 'source': 'C:\\Users\\abubakar\\Desktop\\agentic_notebook\\00_data\\parsing_data\\evs_oil_price_shock.pdf', 'title': '(anonymous)', 'keywords': ''}, page_content="longer in developing markets — complete fleet electrification cannot occur in less than two to\nthree decades even under the most aggressive adoption scenarios. This fleet inertia is the\nprimary reason why even optimistic EV projections show only modest oil demand\ndisplacement through 2030, with the major effects occurring in the 2030–2050 window.\n Under a high-adoption scenario broadly consistent with the IEA's Net Zero by 2050"),
 Document(metadata={'creationdate': '2026-04-18T21:49:03+05:00', 'moddate': '2026-04-18T21

In [30]:
# USER LONG TERM , MESSAGE HISTORY  , CURRNET QUERY , RETIREVED DOCUMENT 

# User Instruction (User Prompt)
User prompt sirf current sawal nahi hota, balke is mein current query ke saath conversation history, retrieved RAG chunks, tool outputs, aur memory bhi shamil ho sakti hai. "Wall of text" se bachne (jo semantic contamination, instruction-data conflation, aur attention dilution paida karta hai) ke liye mukhtalif hisson ko XML tags jaise delimiters mein alag rakha jata hai. KV caching aur "lost in the middle" se bachne ke liye dynamic content aur current query ko aam taur par neeche/aakhir mein rakha jata hai.

### *LETS PASS THIS ENTIRE THING TO LLM , LETS SEE HOW WE DO THIS , SO THAT NO ATTENTION DILUTION, SEMANTIC CONTAMINATION, PATTERN RECOG FAILURE TAKES PLACE.*

In [31]:
# 1. Long-Term Memory Store: Alag-alag mock users (Alex, Sam, Jordan, Priya, Sarah)
# ki identity, response preferences, past chats, aur goals yahan save hain.
long_term_memory_storage = {
    "Alex": {
        "core_identity": "Alex is a highly experienced senior backend engineer currently based out of Berlin, Germany. They have spent the last decade specializing in Python, Go, and building scalable cloud infrastructures on AWS. They possess a very pragmatic, no-nonsense approach to problem-solving, valuing execution speed and efficiency over perfection. They despise boilerplate code and often work late into the night, which is when they usually interact with the system.",
        "user_preferences": "Prefers highly technical, direct responses that entirely skip conversational filler and greetings. Always wants code examples written in Python or Go, with clear comments explaining the 'why' rather than the 'what'. They explicitly requested that UI tools always default to dark mode and that the AI prioritize official documentation links over third-party tutorials.",
        "past_interactions": "Previously spent several multi-hour sessions debugging a complex Docker networking issue and successfully setting up a distributed Redis cache. Last week, we worked together to resolve a stubborn API rate-limiting problem that was crashing their production servers during peak hours.",
        "current_goals": "Currently focused on a massive project to migrate a legacy monolithic application to a modern microservices architecture. They are actively evaluating different API gateway solutions, dealing with complex routing logic, and constantly looking for ways to cut down their monthly AWS billing costs.",
    },
    "Sam": {
        "core_identity": "Sam is a passionate high school history teacher based in Chicago who recently decided to pick up coding as a personal weekend hobby. They are deeply enthusiastic and eager to learn, but they often feel easily overwhelmed by dense technical jargon and rapid pacing. Because their primary background is in the humanities, they learn best through narratives and storytelling.",
        "user_preferences": "Responds best to a warm, friendly, and highly encouraging tone. Needs technical terms to be broken down and explained using real-world, everyday analogies. Prefers step-by-step, guided learning where the AI checks for understanding after each step, rather than receiving the final completed code all at once.",
        "past_interactions": "Successfully built and deployed a basic HTML and CSS portfolio last month, which was a major milestone. We recently moved on to learning JavaScript fundamentals like variables, loops, and DOM manipulation. During our last session, they struggled a bit to conceptualize how arrays and indexes work.",
        "current_goals": "Trying to build a simple, interactive, web-based quiz application for their history students to use before the end of the current school semester. They want to include a feature that tracks the student's high score.",
    },
    "Jordan": {
        "user_preferences": "Jordan consistently uses the platform from a mobile device while commuting on the train. Because of this, they heavily prefer short, highly concise answers that can fit entirely on a single mobile screen. They have explicitly requested that the AI avoid generating long blocks of code, complex markdown tables, or any unnecessary conversational padding."
    },
    "Priya": {
        "core_identity": "Priya is a freelance science fiction author and world-builder living in London. She uses the AI primarily as a collaborative brainstorming partner rather than a technical assistant or search engine. She is highly imaginative, loves exploring abstract philosophical concepts, and enjoys a conversational dynamic that feels like a rapid-fire ping-pong of creative ideas.",
        "user_preferences": "Prefers a highly descriptive, narrative-driven tone with a rich and varied vocabulary. She enjoys when the AI adopts a 'what if' mindset and actively challenges her plot points to help her find logical inconsistencies or plot holes. She absolutely hates generic sci-fi tropes and frequently commands the AI to make its suggestions weirder or more avant-garde.",
        "past_interactions": "Over the last few months, we have developed a massive, detailed lore bible for a cyberpunk novel set on a terraformed moon of Jupiter. We recently finalized the complex political structure of the moon's ruling council and spent a long session debating the socioeconomic implications of zero-gravity agriculture on the lower classes.",
        "current_goals": "Currently trying to write the climactic chapter of her first draft, where the protagonist discovers a major, devastating betrayal by their mentor. She needs heavy assistance pacing the dialogue to build maximum tension without revealing the twist too early in the scene.",
    },
    "Sarah": {
        "core_identity": "Sarah is a Senior Systems Engineer at ACME Global Corporation. She has been with the company for 4 years. She is actively family-planning and preparing to conceive her first child in the near future.",
        "user_preferences": "Prefers straightforward, jargon-free explanations. Since she is not an HR expert, she likes bulleted lists detailing exact entitlements (days/weeks off), financial implications, and step-by-step instructions for navigating the employee portal.",
        "past_interactions": "Previously queried the chatbot to check her current accrued PTO balance, which currently sits at 112 hours. She also recently asked about the open enrollment process to upgrade from the 'Single Employee' to the 'Family' tier on the ACME Gold Health Insurance plan.",
        "current_goals": "Currently focused on understanding family-planning benefits, pre-natal healthcare coverage, and future maternity leave options. She wants to map out how to handle medical appointments and maximize her time off when the time comes.",
    },
}

# 2. Recent Chat History: User aur assistant ke darmiyan pichlay messages
# jo context continuity ke liye bheje jate hain.
chat_history = [
    {"role": "user", "content": "Hi there!"},
    {
        "role": "assistant",
        "content": "Hello Sarah! How can I assist you with your ACME HR benefits today?",
    },
    {
        "role": "user",
        "content": "I am looking into family planning. Does the ACME Gold plan cover pre-natal specialist visits or fertility consultations?",
    },
    {
        "role": "assistant",
        "content": "Yes, Sarah. The ACME Gold plan includes comprehensive pre-natal care and offers up to $10,000 in lifetime family-planning and fertility assistance.",
    },
]

# 4. Active User Selection: Filhal "Sarah" ka memory record fetch kiya ja raha hai.
user_memory = long_term_memory_storage.get("Sarah")

# 5. Delimited Memory Block: Sarah ke memory fields ko XML-style tags mein wrap
# kiya gaya hai taake LLM unhe clear boundaries ke saath parse kar sake.
memory_block = (
    "<user_memory>\n"
    f"<core_identity>\n{user_memory.get('core_identity', '')}\n</core_identity>\n"
    f"<user_preferences>\n{user_memory.get('user_preferences', '')}\n</user_preferences>\n"
    f"<past_interactions>\n{user_memory.get('past_interactions', 'none')}\n</past_interactions>\n"
    f"<current_goals>\n{user_memory.get('current_goals', '')}\n</current_goals>\n"
    "</user_memory>"
)
# 6. Delimited History Block: Pichli conversation ko <conversation_summary> tag
# ke andar rakha gaya hai taake baqi context se mix na ho.
history_block = (
    "<conversation_summary>\n"
    f"{chat_history}\n"
    "</conversation_summary>"
)

# 7. Delimited Question Block: Current user query ko alag tag mein isolate
# kiya gaya hai taake model ka main focus is sawal par rahe.
question_block = f"<user_question>{user_query}</user_question>"

In [32]:
print("Memory Block:\n", memory_block)
print("History Block:\n", history_block)
print("Question Block:\n", question_block)


Memory Block:
 <user_memory>
<core_identity>
Sarah is a Senior Systems Engineer at ACME Global Corporation. She has been with the company for 4 years. She is actively family-planning and preparing to conceive her first child in the near future.
</core_identity>
<user_preferences>
Prefers straightforward, jargon-free explanations. Since she is not an HR expert, she likes bulleted lists detailing exact entitlements (days/weeks off), financial implications, and step-by-step instructions for navigating the employee portal.
</user_preferences>
<past_interactions>
Previously queried the chatbot to check her current accrued PTO balance, which currently sits at 112 hours. She also recently asked about the open enrollment process to upgrade from the 'Single Employee' to the 'Family' tier on the ACME Gold Health Insurance plan.
</past_interactions>
<current_goals>
Currently focused on understanding family-planning benefits, pre-natal healthcare coverage, and future maternity leave options. She w

In [33]:
# Assuming 'found_chunks' is the list of Document objects you provided
doc_tags = []

# CONCEPT: Delimiter Prompting / Context Isolation
# Vector search se jo retrieved chunks aaye hain, unhein structured XML-like tags mein wrap
# kiya ja raha hai taake LLM ko saaf pata chale ke yeh external reference data hai—user ki
# instruction ya system rule nahi. Is se context mixing aur prompt injection/confusion kam hota hai.
for i, doc in enumerate(found_chunks, start=1):
    title = doc.metadata.get("title", "Unknown Title")
    content = doc.page_content
    
    # HANDLING CHUNK BOUNDARIES: Har retrieved chunk ko ek unique 'id' (1, 2, 3...) aur
    # 'title' attribute ke saath alag block banaya gaya hai taake model multiple sources
    # ko easily distinguish aur cite kar sake.
    doc_tags.append(
        f'  <document id="{i}" title="{title}">\n'
        f'  {content}\n'
        f'  </document>'
    )

# HANDLING MULTIPLE CHUNKS: Sab individual <document> tags ko double newline se join kar ke
# ek parent <retrieved_documentation> wrapper mein band kiya gaya hai taake saara retrieved
# context ek unified section ke tor par model ko mile.
docs_block = "<retrieved_documentation>\n" + "\n\n".join(doc_tags) + "\n</retrieved_documentation>"

# HANDLING VERIFICATION: Prompt bhejne se pehle formatted block print kar ke check kiya ja
# raha hai ke tags, indentation, aur retrieved text expected format mein hain ya nahi.
print(docs_block)

<retrieved_documentation>
  <document id="1" title="(anonymous)">
  longer in developing markets — complete fleet electrification cannot occur in less than two to
three decades even under the most aggressive adoption scenarios. This fleet inertia is the
primary reason why even optimistic EV projections show only modest oil demand
displacement through 2030, with the major effects occurring in the 2030–2050 window.
 Under a high-adoption scenario broadly consistent with the IEA's Net Zero by 2050
  </document>

  <document id="2" title="(anonymous)">
  longer in developing markets — complete fleet electrification cannot occur in less than two to
three decades even under the most aggressive adoption scenarios. This fleet inertia is the
primary reason why even optimistic EV projections show only modest oil demand
displacement through 2030, with the major effects occurring in the 2030–2050 window.
 Under a high-adoption scenario broadly consistent with the IEA's Net Zero by 2050
  </documen

In [34]:
# CONCEPT: Multi-Context Prompt Assembly with Explicit Boundaries
# Yahan 4 alag-alag context layers ko ek hi final prompt mein jora ja raha hai:
# 1. memory_block (<user_memory>): User ki profile, preferences, aur past context ke liye.
# 2. history_block (<conversation_summary>): Pichli chat continuity ke liye.
# 3. docs_block (<retrieved_documentation>): Grounded policy facts ke liye.
# 4. <user_question>: Current sawal ko clearly isolate karne ke liye.
# Double newlines ("\n\n") se har section ke darmiyan clear visual/structural separation di gayi hai.
Final_prompt = "\n\n".join([memory_block, history_block, docs_block, f"<user_question>{user_query}</user_question>"])
print(Final_prompt)

<user_memory>
<core_identity>
Sarah is a Senior Systems Engineer at ACME Global Corporation. She has been with the company for 4 years. She is actively family-planning and preparing to conceive her first child in the near future.
</core_identity>
<user_preferences>
Prefers straightforward, jargon-free explanations. Since she is not an HR expert, she likes bulleted lists detailing exact entitlements (days/weeks off), financial implications, and step-by-step instructions for navigating the employee portal.
</user_preferences>
<past_interactions>
Previously queried the chatbot to check her current accrued PTO balance, which currently sits at 112 hours. She also recently asked about the open enrollment process to upgrade from the 'Single Employee' to the 'Family' tier on the ACME Gold Health Insurance plan.
</past_interactions>
<current_goals>
Currently focused on understanding family-planning benefits, pre-natal healthcare coverage, and future maternity leave options. She wants to map out

# system instrction 

System instruction AI application ke "constitution" ki tarah kaam karti hai jo model ke persona, tone, format, goals, aur guardrails define karti hai. PDF ke mutabiq is ke chaar core elements isi tarteeb mein likhe jate hain: Persona & Task, Constraints & Boundaries, Communication Style & Format, aur Audience. Behtar results ke liye ise "right altitude" (clear aur flexible) par rakha jata hai, verbose prose ke bajaye minimal/tagged rakha jata hai, aur iteratively refine kiya jata hai.

In [35]:
system_instruction = """You are a calm, humble, and concise Global HR Assistant for ACME Corporation. Your goal is to provide accurate, empathetic, and personalized guidance directly to the employee.

You will receive four context blocks. Process them as follows:

1. <user_memory>: Employee facts (location, tenure, balances, profile). Use this to personalize your response and apply the correct jurisdiction-specific rules.
2. <conversation_summary>: Recent chat history. Use this to maintain continuity and avoid repeating information or steps the user already knows.
3. <retrieved_documentation>: Official ACME policies. **THIS IS YOUR SINGLE SOURCE OF TRUTH.** Base all facts, rules, and numbers entirely on this block.
4. <user_question>: The immediate query to answer.

### Core Constraints & Behaviors
* **Be Concise & Direct:** Give the employee the exact answer they need without unnecessary fluff or verbose introductions. 
* **Strict Grounding:** If the <retrieved_documentation> does not contain the answer, humbly state that you do not have that specific information. Do not guess, assume, or invent policies.
* **Escalate Gently:** If the scenario involves complex legal/medical intersections, or if the documentation lacks the answer, kindly offer to escalate the case to the HR Benefits Consortium.
* **Formatting:** Structure the response for easy reading. Use bullet points for steps, **bold text** for formal terms (e.g., **Qualifying Family Event**), and markdown tables for comparisons.
* **Tone:** Maintain a warm, calm, and professional demeanor, especially regarding sensitive life events.

Analyze the provided context and output your final, formatted response to the employee."""

In [40]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",  # Groq ka valid model name
    temperature=0.7,
    # GROQ_API_KEY environment variable mein set honi chahiye
)

messages = [
    SystemMessage(content=system_instruction),
    HumanMessage(content=Final_prompt),
]

response = llm.invoke(messages)
print(response.content)

**What the ACME‑provided documentation says**

- The documents note that **electric‑vehicle (EV) adoption will lead to only modest oil‑demand displacement through 2030**.  
- **The larger impact is expected in the 2030‑2050 window**, when fleet electrification accelerates.

**What’s missing**

- The documentation does **not include specific quantitative figures** (e.g., barrels per day or percentage reductions) for either the 2030 horizon or the 2030‑2050 period.

**Next steps**

- If you need exact numbers, I recommend consulting the latest IEA (International Energy Agency) reports or ACME’s internal market‑analysis team, who may have more detailed forecasts.  
- I can help you route this request to the appropriate research group if you’d like.

Let me know how you’d like to proceed!
